# Reproducing an RCCS-Cloud job through the real MCP tool surface

This notebook distills an interactive agent session into a linear, runnable
narrative — but instead of a hand-copied SSH/rsync script, every cluster
interaction below goes through the *same* tool surface (`cloud_mcp.hpc_server`)
the interactive Claude Code / Codex agent uses, via `hpc_agent_core.client`.
A bug fix or regression in the shared transport layer (`middleware.py`) shows
up here too, instead of living in a second, unaudited implementation.

Three caching modes, set once when connecting:

- `mode="live"` — always hits the real cluster; still refreshes the cache as a
  side effect.
- `mode="lazy"` (default) — reuses a local, per-call cache if present. Lets you
  edit a later cell (e.g. how the result is plotted) and re-run "all" without
  resubmitting the job.
- `mode="replay"` — cache-only, raises if anything isn't already recorded. No
  SSH is attempted at all — this is the mode to hand this notebook to someone
  with no cluster account. This repo ships a real recorded cache
  (`.hpc_cache/reproduction-demo/`) from an actual run, so `mode="replay"`
  works immediately after cloning, with zero cluster access.

**A `mode="lazy"` run is not proof this notebook still reproduces end-to-end**
— a stale cache happily papers over a since-broken call. Re-run with
`mode="live"` at least once after any change to `cloud_mcp` or `hpc-agent-core`
before trusting this as a validated artifact (the same discipline as the read-only/job
smoke tiers in `tests/smoke.py` — see `PORTING.md` §9).

This uses `connect_sync` — no `await`, no `async with` spread across cells.
(`hpc_agent_core.client` also has a plain `connect()` async context manager,
for concurrent or already-async use; `connect_sync` is the one built for a
notebook's cell-by-cell, blocking narrative.)

**Setup**: `pip install -e ../server` (this repo's `cloud_mcp` package, which
pins `hpc-agent-core==0.5.0` — the version this notebook's `client.py` API
ships in).


In [1]:
from hpc_agent_core.client import connect_sync, dev_params

CACHE_DIR = "./.hpc_cache/reproduction-demo"

hpc = connect_sync(dev_params("cloud_mcp", "hpc_server"), mode="lazy", cache_dir=CACHE_DIR)
print("connected (mode=lazy)")

connected (mode=lazy)

## 1. Stage a scoped remote directory

Everything this notebook touches on the cluster lives under
`~/agent/notebook-demo/`, mirroring the family's "visible directory" convention
(see `AGENTS.md`) so it's easy to find and clean up by hand if needed.

In [2]:
created = hpc.fs_mkdir(path="agent/notebook-demo")
print(created)

created: /hs/work0/home/users/william.dawson/agent/notebook-demo

## 2. Submit the job

A tiny, dependency-free compute job: sum of squares 1..100,000, computed with
plain `awk` (no Python/compiler needed on the compute node) and written to
`result.json`. `directory` pins the job's working directory so we know exactly
where the output lands.

Partition choice, honestly: this cluster's default CPU partition (`genoa`) was
saturated with other users' multi-hour jobs when this was recorded (`squeue`
showed ours `PENDING (Resources)` indefinitely) — a real, live, shared cluster,
not a mock. `fx700` (Fujitsu A64FX, aarch64) had 20 idle nodes at the time, so
that's what actually ran this. `sinfo` is worth a glance before picking a
partition for anything latency-sensitive.

In [3]:
spec = {
    "name": "notebook-demo",
    "executable": (
        "module load system/fx700 FJSVstclanga\n"
        "sum=$(awk 'BEGIN{for(i=1;i<=100000;i++)s+=i*i; print s}')\n"
        "printf '{\"sum_of_squares\": %s, \"host\": \"%s\"}' \"$sum\" \"$(hostname)\" "
        "> result.json\n"
    ),
    "directory": "agent/notebook-demo",
    "attributes": {"duration": "00:05:00", "queue_name": "fx700"},
    "resources": {"node_count": 1},
}
job = hpc.submit_job(spec=spec)
print(job)

{'job_id': '266362', 'script_path': '/hs/work0/home/users/william.dawson/agent/jobs/notebook-demo-20260804-154917.sh'}

## 3. Wait for it to finish

`wait_for_job` polls `get_job_status` until a terminal state, and — unlike the
generic per-call cache — only caches the *terminal* result, never an
intermediate PENDING/RUNNING poll (see its docstring for why that distinction
matters). On this run it took about a second once scheduled.

In [4]:
status = hpc.wait_for_job(job["job_id"])
print(status)

{'id': '266362', 'status': {'state': 'completed', 'time': 1785826159.0, 'message': None, 'exit_code': 0, 'meta_data': {'native_state': 'COMPLETED', 'name': 'notebook-demo', 'partition': 'fx700', 'elapsed': '00:00:01', 'start_time': '2026-08-04T15:49:18', 'end_time': '2026-08-04T15:49:19', 'nodes': 'fx12', 'workdir': '/home/users/william.dawson/agent/notebook-demo'}}, 'job_spec': None}

## 4. Pull back the result

`fs_download` is the one helper with bespoke caching: a cache hit here
materializes the *actual bytes* into `local_path` (copied from the cache
directory), not just a metadata dict claiming success — otherwise `mode="replay"`
from a fresh clone would report `verified: true` while `downloads/result.json`
silently never got written.

In [5]:
import json
from pathlib import Path

Path("downloads").mkdir(exist_ok=True)
dl = hpc.fs_download(path="agent/notebook-demo/result.json", local_path="./downloads/result.json")
print(dl)

result = json.loads(Path("./downloads/result.json").read_text())
print(result)

expected = sum(i * i for i in range(1, 100_001))
assert result["sum_of_squares"] == expected
print(f"sum_i=1^100000 i^2 = {expected:,} — matches the value computed on {result['host']}")

{'local_path': './downloads/result.json', 'bytes': 72, 'sha256': 'ffa06c6ab153609df05e45d0565b2c8311eb8f3c4ecfecbc5e5383f3167722a8', 'verified': True, 'transport': 'rsync'}
{'sum_of_squares': 333338333350000, 'host': 'fx12.cloud.r-ccs.riken.jp'}
sum_i=1^100000 i^2 = 333,338,333,350,000 — matches the value computed on fx12.cloud.r-ccs.riken.jp

## 5. Prove the caching modes

Re-running the submit cell above with the *same* `spec` is a cache hit — no
second job gets queued:

In [6]:
job_again = hpc.submit_job(spec=spec)
assert job_again == job
print("same job_id from cache, no new submission:", job_again["job_id"])

same job_id from cache, no new submission: 266362

Now the real test: close this connection, open a **brand new** one in
`mode="replay"` pointed at the same cache directory — simulating a fresh
`git clone` with no cluster account at all — and replay the entire narrative:

In [7]:
hpc.close()

hpc_replay = connect_sync(dev_params("cloud_mcp", "hpc_server"), mode="replay", cache_dir=CACHE_DIR)

replayed_job = hpc_replay.submit_job(spec=spec)
replayed_status = hpc_replay.wait_for_job(replayed_job["job_id"])
replayed_dl = hpc_replay.fs_download(
    path="agent/notebook-demo/result.json", local_path="./downloads/result-replay.json"
)
replayed_result = json.loads(Path("./downloads/result-replay.json").read_text())

assert replayed_job == job
assert replayed_status == status
assert replayed_result == result
print("replay mode reproduced the full narrative -- job id, terminal status, and the")
print("actual downloaded file's bytes -- without opening a single SSH connection.")

hpc_replay.close()

replay mode reproduced the full narrative -- job id, terminal status, and the
actual downloaded file's bytes -- without opening a single SSH connection.

---

**Provenance**: job `266362`, node `fx12`, submitted and completed
2026-08-04T15:49:18Z–15:49:19Z on RCCS-Cloud, recorded with
`hpc-agent-core` `0.5.0` (pre-release) + `cloud_mcp` (RCCS-CloudAgent). The
job id/host/checksums above are real output from that run, not authored — see
`.hpc_cache/reproduction-demo/` for the raw recorded calls.